In [ ]:
%matplotlib inline
from pathlib import Path
import shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Epochs for each mini-batch optimizer when IS_UNITTEST = False.
FULL_RUN_EPOCHS = 5
# Full-dataset gradient-descent steps when IS_UNITTEST = False.
FULL_RUN_GD_STEPS = 100
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)


seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
src = source.code.src
answer = {}


# Comparing Optimizers

Every optimizer answers the same question: given the gradient, how should
the weights move? In this notebook we train **the same network, from the
same initial weights, on the same data** with:

1. full-dataset gradient descent, written by hand;
2. mini-batch gradient descent;
3. momentum;
4. AdaGrad;
5. Adam;
6. Lion;
7. Muon.

After that we add three tools that work on top of any optimizer:
**Lookahead**, an **exponential moving average of the weights**, and
**reducing the learning rate on a plateau**.

You write gradient descent by hand once. After that, each optimizer is
only its update formula: open [src/optimizers.py](../../source/code/src/optimizers.py).
The class `Optimizer` there already loops over the parameters and keeps
a `state` dictionary for each one; you fill in `update(param, grad, state)`,
which changes one parameter in place.

The small run (`IS_UNITTEST = True`) trains on tiny generated data, so its
accuracy means nothing; it only checks your formulas. Set
`IS_UNITTEST = False` in the first cell to compare the optimizers on
real MNIST.


# Setup

The network is `SimpleFCNN` from the Fully Connected NNs chapter: one
hidden layer of 64 units. `make_model()` reseeds the random generators
first, so every optimizer starts from exactly the same weights, and every
training run sees the batches in the same order.


In [ ]:
src.utils.seed.seed_all(0)

MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)

n_epochs = 2 if IS_UNITTEST else FULL_RUN_EPOCHS
n_gd_steps = 5 if IS_UNITTEST else FULL_RUN_GD_STEPS
batch_size = 32 if IS_UNITTEST else 64


def make_model():
    src.utils.seed.seed_all(0)
    return src.models.feedforward.simple_fcnn.SimpleFCNN(
        [28 * 28, 64], n_classes=10, activation=torch.nn.ReLU)


def cross_entropy(batch):
    return torch.nn.functional.cross_entropy(
        batch['signals']['output'], batch['data']['label'])


def accuracy(batch):
    correct = (batch['postprocessed']['class'] == batch['data']['label']).float().sum()
    return correct, len(batch['data']['label'])


losses = {'ce': cross_entropy}
metrics = {'accuracy': accuracy}

valid_dl = torch.utils.data.DataLoader(MNIST_valid, batch_size=256, shuffle=False)
# The whole training set as one batch, for full-dataset gradient descent.
full_batch = next(iter(torch.utils.data.DataLoader(MNIST_train, batch_size=len(MNIST_train))))


def make_train_loader():
    return torch.utils.data.DataLoader(
        MNIST_train, batch_size=batch_size, shuffle=True, drop_last=True,
        generator=torch.Generator().manual_seed(0))


def train(name, make_optimizer, epochs=n_epochs, train_dl=None, make_scheduler=None):
    """Train a fresh model; return the model, the optimizer and the history."""
    model = make_model()
    optimizer = make_optimizer(model.parameters())
    scheduler = make_scheduler(optimizer) if make_scheduler else None
    history = src.train_loop.train_model(
        model, epochs, train_dl or make_train_loader(), valid_dl, losses,
        optimizer, metrics=metrics, scheduler=scheduler, run_name=name)
    return model, optimizer, {
        'train_loss': history['train_loss']['ce'],
        'valid_loss': history['valid_loss']['ce'],
        'valid_accuracy': history['valid_metrics']['accuracy'],
    }


histories = {}
answer['steps'] = {}
answer['training'] = {}


# Task 1: Gradient Descent by Hand

Full-dataset gradient descent uses the gradient of the loss over **all**
training images at every step:

$$\theta \leftarrow \theta - \eta \, \nabla_\theta L(\theta).$$

Complete the loop below. After `model(batch)` and the loss:

- call `loss.backward()` to get `param.grad` for every parameter;
- inside `torch.no_grad()`, move every parameter: `param -= learning_rate * param.grad`;
- clear the gradient (`param.grad = None`), otherwise the next
  `backward()` adds to it.


In [ ]:
# Edit this cell in task.ipynb.
learning_rate = 0.1
model = make_model()
gd_by_hand_loss = []

for step in range(n_gd_steps):
    batch = {'data': full_batch}
    model(batch)
    loss = losses['ce'](batch)
    gd_by_hand_loss.append(float(loss))

    ## YOUR CODE HERE

print('loss at each step:', np.round(gd_by_hand_loss, 4))
answer['gd_by_hand'] = gd_by_hand_loss


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Gradient Descent by Hand', answer,
    prefixes=('gd_by_hand',),
    is_small_run=IS_UNITTEST)


# Task 2: The Same Step as a Formula

From now on the loop is written once, in `Optimizer.step` and
`train_model`. An optimizer only says how **one** parameter moves.

Implement `GradientDescent.update` in [src/optimizers.py](../../source/code/src/optimizers.py):

$$\theta \leftarrow \theta - \eta g.$$

`check_steps` runs three steps on a tiny fixed problem, so a mistake in
the formula shows up before any training. Then we train twice with the
same formula:

- **full-dataset** gradient descent: one step per pass over the data
  (it should repeat your Task 1 losses);
- **mini-batch** gradient descent: one step per batch of
  `batch_size` images, many steps per pass.


In [ ]:
# Edit: ../../source/code/src/optimizers.py (GradientDescent.update)
answer['steps']['gradient_descent'] = src.optimizers.check_steps(
    lambda params: src.optimizers.GradientDescent(params, lr=0.1))

model, optimizer, histories['full-dataset GD'] = train(
    'full-dataset GD', lambda params: src.optimizers.GradientDescent(params, lr=0.1),
    epochs=n_gd_steps, train_dl=[full_batch])
print('Same losses as in Task 1:',
      np.allclose(histories['full-dataset GD']['train_loss'], gd_by_hand_loss, atol=1e-5))

model, optimizer, histories['mini-batch GD'] = train(
    'mini-batch GD', lambda params: src.optimizers.GradientDescent(params, lr=0.1))

answer['training']['full_dataset_gd'] = histories['full-dataset GD']
answer['training']['minibatch_gd'] = histories['mini-batch GD']


In [ ]:
course_utils.checker.print_progress(
    'Task 2: The Same Step as a Formula', answer,
    prefixes=('steps -> gradient_descent', 'training -> full_dataset_gd', 'training -> minibatch_gd'),
    is_small_run=IS_UNITTEST)


# Task 3: Momentum

Gradient descent forgets the previous steps. Momentum keeps a running sum
of gradients, a velocity $v$, and moves along it:

$$v \leftarrow \beta v + g, \qquad \theta \leftarrow \theta - \eta v.$$

Directions that agree from step to step add up; noisy directions cancel.
Implement `Momentum.update` in [src/optimizers.py](../../source/code/src/optimizers.py).
`state['velocity']` starts at zero; update it in place.


In [ ]:
# Edit: ../../source/code/src/optimizers.py (Momentum.update)
answer['steps']['momentum'] = src.optimizers.check_steps(
    lambda params: src.optimizers.Momentum(params, lr=0.1, beta=0.9))
model, optimizer, histories['momentum'] = train(
    'momentum', lambda params: src.optimizers.Momentum(params, lr=0.01, beta=0.9))
answer['training']['momentum'] = histories['momentum']


In [ ]:
course_utils.checker.print_progress(
    'Task 3: Momentum', answer,
    prefixes=('steps -> momentum', 'training -> momentum'),
    is_small_run=IS_UNITTEST)


# Task 4: AdaGrad

One learning rate for all weights is a compromise. AdaGrad gives every
coordinate its own step: it sums the squared gradients $s$ and divides
by $\sqrt{s}$, so coordinates with large gradients take smaller steps:

$$s \leftarrow s + g^2, \qquad \theta \leftarrow \theta - \eta \frac{g}{\sqrt{s} + \varepsilon}.$$

The sum only grows, so AdaGrad's steps keep shrinking. Implement
`AdaGrad.update` in [src/optimizers.py](../../source/code/src/optimizers.py).


In [ ]:
# Edit: ../../source/code/src/optimizers.py (AdaGrad.update)
answer['steps']['adagrad'] = src.optimizers.check_steps(
    lambda params: src.optimizers.AdaGrad(params, lr=0.1))
model, optimizer, histories['AdaGrad'] = train(
    'AdaGrad', lambda params: src.optimizers.AdaGrad(params, lr=0.05))
answer['training']['adagrad'] = histories['AdaGrad']


In [ ]:
course_utils.checker.print_progress(
    'Task 4: AdaGrad', answer,
    prefixes=('steps -> adagrad', 'training -> adagrad'),
    is_small_run=IS_UNITTEST)


# Task 5: Adam

Adam combines both ideas, with running **averages** instead of sums:
$m$ averages the gradients (momentum), $v$ averages their squares (the
per-coordinate scale, like AdaGrad, but it can also grow back):

$$m \leftarrow \beta_1 m + (1 - \beta_1) g, \qquad v \leftarrow \beta_2 v + (1 - \beta_2) g^2.$$

Both start at zero, so early averages are too small. Dividing by
$1 - \beta^t$, where $t$ is the step number, corrects that:

$$\hat m = \frac{m}{1 - \beta_1^t}, \qquad \hat v = \frac{v}{1 - \beta_2^t}, \qquad \theta \leftarrow \theta - \eta \frac{\hat m}{\sqrt{\hat v} + \varepsilon}.$$

Implement `Adam.update` in [src/optimizers.py](../../source/code/src/optimizers.py);
`step` already counts the steps.


In [ ]:
# Edit: ../../source/code/src/optimizers.py (Adam.update)
answer['steps']['adam'] = src.optimizers.check_steps(
    lambda params: src.optimizers.Adam(params, lr=0.1))
model, optimizer, histories['Adam'] = train(
    'Adam', lambda params: src.optimizers.Adam(params, lr=1e-3))
answer['training']['adam'] = histories['Adam']


In [ ]:
course_utils.checker.print_progress(
    'Task 5: Adam', answer,
    prefixes=('steps -> adam', 'training -> adam'),
    is_small_run=IS_UNITTEST)


# Task 6: Lion

Lion was found by an automated search over update rules (Chen et al.,
2023). It keeps only a momentum $m$ and uses just the **sign** of a mix of
$m$ and the new gradient, so every coordinate moves by exactly $\eta$:

$$\theta \leftarrow \theta - \eta \, \mathrm{sign}\big(\beta_1 m + (1 - \beta_1) g\big), \qquad m \leftarrow \beta_2 m + (1 - \beta_2) g.$$

The order matters: the step uses the old $m$. Because all steps have the
same size, Lion needs a learning rate about 3 to 10 times smaller than
Adam's. Implement `Lion.update` in [src/optimizers.py](../../source/code/src/optimizers.py).


In [ ]:
# Edit: ../../source/code/src/optimizers.py (Lion.update)
answer['steps']['lion'] = src.optimizers.check_steps(
    lambda params: src.optimizers.Lion(params, lr=0.01))
model, optimizer, histories['Lion'] = train(
    'Lion', lambda params: src.optimizers.Lion(params, lr=3e-4))
answer['training']['lion'] = histories['Lion']


In [ ]:
course_utils.checker.print_progress(
    'Task 6: Lion', answer,
    prefixes=('steps -> lion', 'training -> lion'),
    is_small_run=IS_UNITTEST)


# Task 7: Muon

Muon (Jordan et al., 2024) treats a weight **matrix** as a whole. It
keeps a momentum $m$ and replaces it by the nearest matrix whose singular
values are all about 1, so the step changes the layer equally in every
direction, instead of mostly along the few largest ones:

$$m \leftarrow \beta m + (1 - \beta) g, \qquad \theta \leftarrow \theta - \eta \cdot \mathrm{scale} \cdot \mathrm{orthogonalize}(m).$$

`orthogonalize` (a few Newton–Schulz iterations) and `scale` are given.
Parameters that are not matrices, the biases, are updated by Adam.
Implement the matrix case of `Muon.update` in
[src/optimizers.py](../../source/code/src/optimizers.py).


In [ ]:
# Edit: ../../source/code/src/optimizers.py (Muon.update)
answer['steps']['muon'] = src.optimizers.check_steps(
    lambda params: src.optimizers.Muon(params, lr=0.02, adam_lr=0.1))
model, optimizer, histories['Muon'] = train(
    'Muon', lambda params: src.optimizers.Muon(params, lr=0.01))
answer['training']['muon'] = histories['Muon']


In [ ]:
course_utils.checker.print_progress(
    'Task 7: Muon', answer,
    prefixes=('steps -> muon', 'training -> muon'),
    is_small_run=IS_UNITTEST)


# Comparison

Each curve is one optimizer, from the same initial weights. The x axis
counts passes over the training data. Full-dataset gradient descent makes
one step per pass; mini-batch gradient descent makes one step per batch,
hundreds per pass. The first plot compares the two (log scale, because
full-dataset gradient descent needs many more passes); the second compares
the mini-batch optimizers.


In [ ]:
def plot_histories(names, log_x=False):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for name in names:
        epochs = range(1, len(histories[name]['valid_loss']) + 1)
        axes[0].plot(epochs, histories[name]['valid_loss'], label=name)
        axes[1].plot(epochs, histories[name]['valid_accuracy'], label=name)
    axes[0].set_title('validation loss')
    axes[1].set_title('validation accuracy')
    for ax in axes:
        ax.set_xlabel('passes over the training data')
        if log_x:
            ax.set_xscale('log')
        ax.legend()
    plt.show()


plot_histories(['full-dataset GD', 'mini-batch GD'], log_x=True)
plot_histories(['mini-batch GD', 'momentum', 'AdaGrad', 'Adam', 'Lion', 'Muon'])


# Task 8: Lookahead

Lookahead (Zhang et al., 2019) wraps any optimizer. The inner optimizer
moves the weights as usual (the *fast* weights). Every $k$ steps, a second
copy (the *slow* weights) moves part of the way towards them, and the
fast weights restart from there:

$$\theta_{slow} \leftarrow \theta_{slow} + \alpha (\theta_{fast} - \theta_{slow}), \qquad \theta_{fast} \leftarrow \theta_{slow}.$$

Implement the slow-weight update in `Lookahead.step` in
[src/optimizers.py](../../source/code/src/optimizers.py); the reset is given.


In [ ]:
# Edit: ../../source/code/src/optimizers.py (Lookahead.step)
answer['steps']['lookahead'] = src.optimizers.check_steps(
    lambda params: src.optimizers.Lookahead(
        src.optimizers.GradientDescent(params, lr=0.1), k=2, alpha=0.5))
model, optimizer, histories['Lookahead(Adam)'] = train(
    'Lookahead(Adam)', lambda params: src.optimizers.Lookahead(
        src.optimizers.Adam(params, lr=1e-3), k=5, alpha=0.5))
answer['training']['lookahead'] = histories['Lookahead(Adam)']


In [ ]:
course_utils.checker.print_progress(
    'Task 8: Lookahead', answer,
    prefixes=('steps -> lookahead', 'training -> lookahead'),
    is_small_run=IS_UNITTEST)


# Task 9: Exponential Moving Average of the Weights

The weights at the last step carry the noise of the last few batches. An
average of recent weights is often better. `WeightAverage` keeps, next to
training, an exponential moving average of every parameter:

$$\bar\theta \leftarrow \gamma \bar\theta + (1 - \gamma) \theta.$$

Training still moves $\theta$; we evaluate with $\bar\theta$. Implement
the average in `WeightAverage.step` in [src/optimizers.py](../../source/code/src/optimizers.py).


In [ ]:
# Edit: ../../source/code/src/optimizers.py (WeightAverage.step)
answer['steps']['weight_average'] = src.optimizers.check_steps(
    lambda params: src.optimizers.WeightAverage(
        src.optimizers.GradientDescent(params, lr=0.1), decay=0.5))
model, weight_average, histories['Adam + weight EMA'] = train(
    'Adam + weight EMA', lambda params: src.optimizers.WeightAverage(
        src.optimizers.Adam(params, lr=1e-3), decay=0.99))


def evaluate(model):
    model.eval()
    total_loss, correct = 0.0, 0.0
    with torch.no_grad():
        for data in valid_dl:
            batch = {'data': data}
            model(batch)
            total_loss += float(losses['ce'](batch)) * len(data['label'])
            correct += float(metrics['accuracy'](batch)[0])
    return total_loss / len(MNIST_valid), correct / len(MNIST_valid)


last_loss, last_accuracy = evaluate(model)
with weight_average.averaged_weights():
    average_loss, average_accuracy = evaluate(model)
print(f'last weights:     loss {last_loss:.4f}, accuracy {last_accuracy:.4f}')
print(f'averaged weights: loss {average_loss:.4f}, accuracy {average_accuracy:.4f}')

answer['training']['weight_average'] = histories['Adam + weight EMA']
answer['training']['weight_average_eval'] = {
    'loss': average_loss, 'accuracy': average_accuracy}


In [ ]:
course_utils.checker.print_progress(
    'Task 9: Exponential Moving Average of the Weights', answer,
    prefixes=('steps -> weight_average', 'training -> weight_average'),
    is_small_run=IS_UNITTEST)


# Task 10: Reduce the Learning Rate on a Plateau

A large learning rate makes fast progress at first and then jumps around
the minimum. `ReduceLROnPlateau` watches the validation loss after every
epoch. It already counts the epochs without improvement; when there were
more than `patience` of them in a row, it should cut the learning rate
(an epoch counts as an improvement only if it lowers the loss by more than
`threshold`, as a fraction):

$$\eta \leftarrow \text{factor} \cdot \eta.$$

Implement that line in `ReduceLROnPlateau.step` in
[src/optimizers.py](../../source/code/src/optimizers.py). The check below feeds it a fixed
sequence of losses; then we train mini-batch gradient descent with it.


In [ ]:
# Edit: ../../source/code/src/optimizers.py (ReduceLROnPlateau.step)
plateau = src.optimizers.ReduceLROnPlateau(
    src.optimizers.GradientDescent([], lr=1.0), factor=0.5, patience=1)
plateau_lrs = []
for loss in [1.0, 0.8, 0.85, 0.9, 0.7, 0.75, 0.8, 0.9]:
    plateau.step(loss)
    plateau_lrs.append(plateau.optimizer.lr)
print('learning rate after each epoch:', plateau_lrs)
answer['steps']['plateau_lrs'] = plateau_lrs

model, optimizer, histories['GD + plateau'] = train(
    'GD + plateau', lambda params: src.optimizers.GradientDescent(params, lr=0.1),
    make_scheduler=lambda optimizer: src.optimizers.ReduceLROnPlateau(
        optimizer, factor=0.5, patience=0, threshold=0.1))
print('final learning rate:', optimizer.lr)
answer['training']['plateau'] = histories['GD + plateau']
answer['training']['plateau_final_lr'] = optimizer.lr


In [ ]:
course_utils.checker.print_progress(
    'Task 10: Reduce the Learning Rate on a Plateau', answer,
    prefixes=('steps -> plateau_lrs', 'training -> plateau'),
    is_small_run=IS_UNITTEST)


# Comparison with the Wrappers


In [ ]:
# The weight average is evaluated once, at the end (Task 9), so it is not here.
plot_histories(['mini-batch GD', 'GD + plateau', 'Adam', 'Lookahead(Adam)'])


# Paths on a Curved Valley

The Rosenbrock function $f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$ has its
minimum at $(1, 1)$, at the bottom of a long, narrow, curved valley: easy
to fall into, slow to follow. Each path starts at $(-1.5, 2)$ and makes
300 steps with your optimizers. Watch how far each one gets along the
valley: AdaGrad's steps shrink until it stalls, Lion moves in steps of a
fixed size, momentum keeps its speed around the bend.


In [ ]:
def rosenbrock(point):
    x, y = point[0, 0], point[0, 1]
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2


def optimizer_path(make_optimizer, n_steps=300):
    point = torch.tensor([[-1.5, 2.0]], requires_grad=True)
    optimizer = make_optimizer([point])
    path = [point.detach().clone()]
    for _ in range(n_steps):
        optimizer.zero_grad()
        rosenbrock(point).backward()
        optimizer.step()
        path.append(point.detach().clone())
    return torch.cat(path).numpy()


paths = {
    'gradient descent': optimizer_path(lambda p: src.optimizers.GradientDescent(p, lr=1e-3)),
    'momentum': optimizer_path(lambda p: src.optimizers.Momentum(p, lr=1e-3, beta=0.9)),
    'AdaGrad': optimizer_path(lambda p: src.optimizers.AdaGrad(p, lr=0.3)),
    'Adam': optimizer_path(lambda p: src.optimizers.Adam(p, lr=0.1)),
    'Lion': optimizer_path(lambda p: src.optimizers.Lion(p, lr=0.01)),
    'Muon': optimizer_path(lambda p: src.optimizers.Muon(p, lr=0.02)),
}

xs, ys = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))
values = (1 - xs) ** 2 + 100 * (ys - xs ** 2) ** 2
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.contour(xs, ys, np.log1p(values), levels=30, cmap='Greys')
for name, path in paths.items():
    line, = ax.plot(path[:, 0], path[:, 1], linewidth=1, alpha=0.8)
    # A large dot marks where each optimizer is after the last step.
    ax.plot(path[-1, 0], path[-1, 1], 'o', color=line.get_color(), markersize=9,
            markeredgecolor='black', label=f'{name}: f = {rosenbrock(torch.tensor(path[-1:])):.2f}')
ax.plot(1, 1, 'k*', markersize=12)
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 3)
ax.legend(title='after 300 steps')
plt.show()


# What to Try Next

- Set `IS_UNITTEST = False` and compare the optimizers on real MNIST.
- Change one learning rate at a time: which optimizers are the most
  sensitive to it?
- Wrap Muon or Lion in Lookahead, or average their weights.


In [ ]:
course_utils.checker.save_answer(answer)
